# Decodo + LlamaIndex RAG Pipeline

This notebook demonstrates a full Retrieval-Augmented Generation (RAG) pipeline
built with the `decodo-llamaindex` package.

**What we will do:**
1. Scrape a set of URLs with `DecodoWebReader`
2. Index the resulting documents into a `VectorStoreIndex`
3. Query the index with natural-language questions

---

## Prerequisites

```bash
pip install decodo-llamaindex llama-index-llms-openai llama-index-embeddings-openai
```

Set environment variables:

```bash
export DECODO_API_TOKEN="your-decodo-token"
export OPENAI_API_KEY="your-openai-key"
```

In [ ]:
import os

# Verify that required environment variables are set
for var in ("DECODO_API_TOKEN", "OPENAI_API_KEY"):
    value = os.environ.get(var)
    if not value:
        raise EnvironmentError(f"Please set the {var} environment variable.")
    masked = value[:4] + "*" * (len(value) - 4)
    print(f"{var} = {masked}")

## Step 1 — Scrape URLs with DecodoWebReader

In [ ]:
from decodo_llamaindex import DecodoWebReader

# URLs to ingest into the knowledge base
URLS = [
    "https://en.wikipedia.org/wiki/Large_language_model",
    "https://en.wikipedia.org/wiki/Retrieval-augmented_generation",
    "https://en.wikipedia.org/wiki/Transformer_(deep_learning_architecture)",
]

reader = DecodoWebReader(api_token=os.environ["DECODO_API_TOKEN"])

print(f"Scraping {len(URLS)} URLs via the Decodo API ...")
documents = reader.load_data(URLS)

print(f"\nLoaded {len(documents)} document(s).")
for doc in documents:
    url = doc.metadata.get("url", "unknown")
    status = doc.metadata.get("status_code", "?")
    chars = len(doc.text)
    print(f"  [{status}] {url}  —  {chars:,} chars")

### Inspect a document

In [ ]:
# Print the first 800 characters of the first document
first_doc = documents[0]
print("URL:", first_doc.metadata["url"])
print("Status:", first_doc.metadata["status_code"])
print()
print(first_doc.text[:800], "...")

## Step 2 — Configure LLM and Embedding Model

In [ ]:
from llama_index.core.settings import Settings
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI

Settings.llm = OpenAI(model="gpt-4o", temperature=0)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

print("LLM:        ", Settings.llm.model)
print("Embed model:", Settings.embed_model.model_name)

## Step 3 — Build a VectorStoreIndex

In [ ]:
from llama_index.core import VectorStoreIndex

print("Building VectorStoreIndex (this will call the OpenAI embeddings API) ...")
index = VectorStoreIndex.from_documents(documents, show_progress=True)
print("Index built.")

## Step 4 — Create a Query Engine and Ask Questions

In [ ]:
query_engine = index.as_query_engine(
    similarity_top_k=3,    # retrieve top 3 most relevant chunks
    response_mode="compact",
)

print("Query engine ready.")

In [ ]:
QUESTIONS = [
    "What is retrieval-augmented generation (RAG) and how does it work?",
    "Explain the transformer architecture and its key components.",
    "What are the main differences between encoder-only, decoder-only, "
    "and encoder-decoder language models?",
]

for q in QUESTIONS:
    print("\n" + "=" * 70)
    print(f"Q: {q}")
    print("=" * 70)
    response = query_engine.query(q)
    print(response)

## Step 5 — Chat Interface (optional)

LlamaIndex also provides a stateful chat engine that maintains conversation history.

In [ ]:
chat_engine = index.as_chat_engine(
    chat_mode="condense_plus_context",
    similarity_top_k=3,
    verbose=False,
)

# First turn
response1 = chat_engine.chat("What is a large language model?")
print("Turn 1:", response1)

# Follow-up (the engine remembers the previous turn)
response2 = chat_engine.chat("How does fine-tuning relate to what you just described?")
print("\nTurn 2:", response2)

## Bonus — Load Search Results into the Index

You can also seed the index with live search results using `DecodoSearchReader`.

In [ ]:
from decodo_llamaindex import DecodoSearchReader

search_reader = DecodoSearchReader(api_token=os.environ["DECODO_API_TOKEN"])

# Fetch Google results for a topic
search_docs = search_reader.load_data(
    "LlamaIndex VectorStoreIndex tutorial 2025",
    engine="google",
)

print(f"Loaded {len(search_docs)} search-result document(s).")
for doc in search_docs:
    print(f"  query={doc.metadata['query']!r}  chars={len(doc.text):,}")

In [ ]:
# Merge search documents into the existing index
for doc in search_docs:
    index.insert(doc)

print("Search documents inserted into index.")

# Query the enriched index
enriched_engine = index.as_query_engine(similarity_top_k=4)
result = enriched_engine.query(
    "What are the latest best practices for building RAG pipelines with LlamaIndex?"
)
print(result)